# OceanEmbed: Historical Dataset Expansion (2018–2025)
**Project Code:** SIH26066 | **Focus:** 3D Ocean Temperature Reconstruction (North Indian Ocean)

This notebook executes and verifies the **Historical Dataset Expansion**:
1. **Additional Genuine Argo Floats**: Expands the in-situ ocean observation library with **7 additional genuine profiling floats** from the official INCOIS Global Data Assembly Centre (`2902268`, `2902279`, `2902280`, `2902283`, `1902670`, `1902672`, `1902673`).
2. **Additional Synchronized Satellite Suite**: Downloads a 3rd distinct historical seasonal window (**January 15–18, 2024: Northeast Winter Monsoon**) covering Sea Surface Temperature (SST), Sea Level Anomaly (SLA), and Geostrophic Surface Currents ($U, V$).
3. **Duplicate Prevention**: Reuses already downloaded files; downloads only net-new files.
4. **Data Preservation**: Preserves all existing raw NetCDF and processed files.
5. **Quality Control & Validation**: Inspects QC flags (`'1'`, `'2'`, `'4'`), coordinate bounds, depth ranges, and valid measurements.
6. **Collocation Matching**: Evaluates satellite–Argo matching across all 3 seasonal observation epochs across Tier 0 (Exact), Tier 1 (Synoptic), Tier 2 (Standard), and Tier 3 (Mesoscale).

> [!NOTE]
> Strictly observational data collection, QA audit, and matching. No preprocessing or model training is conducted in this notebook.

In [1]:
import os
import sys
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import requests
import xarray as xr
import netCDF4
import json

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# Define paths
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
ARGO_DIR = DATA_RAW / "argo"
SST_DIR = DATA_RAW / "sst"
SLA_DIR = DATA_RAW / "sea_level"
CURRENTS_DIR = DATA_RAW / "currents"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Environment Paths:")
print(f"  Project Root:   {PROJECT_ROOT.resolve()}")
print(f"  Raw Data:       {DATA_RAW.resolve()}")
print(f"  Processed Data: {DATA_PROCESSED.resolve()}")

Environment Paths:
  Project Root:   C:\Users\HP\OneDrive\Desktop(1)\Project395\backend
  Raw Data:       C:\Users\HP\OneDrive\Desktop(1)\Project395\backend\data\raw
  Processed Data: C:\Users\HP\OneDrive\Desktop(1)\Project395\backend\data\processed


In [2]:
def download_file_safely(url: str, dest_path: Path, description: str) -> Path:
    """Downloads a file only if it does not already exist, preventing duplicate downloads."""
    if dest_path.exists() and dest_path.stat().st_size > 10_000:
        size_kb = dest_path.stat().st_size / 1024
        print(f"  [EXISTS - SKIPPED] {dest_path.name} ({size_kb:.1f} KB) - Duplicate avoided.")
        return dest_path

    dest_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"  [+] Downloading {description}...")
    print(f"      URL: {url}")
    try:
        r = requests.get(url, stream=True, timeout=60, allow_redirects=True)
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
        size_kb = dest_path.stat().st_size / 1024
        print(f"      [SUCCESS] Saved: {dest_path.name} ({size_kb:.1f} KB)")
        return dest_path
    except Exception as e:
        print(f"      [ERROR] Failed to download {description}: {e}")
        if dest_path.exists():
            dest_path.unlink()
        return None

## 1. Additional Genuine Argo Float Ingestion (2018–2025)

We ingest 7 additional active and historical profiling floats deployed by INCOIS in the North Indian Ocean:
- `2902268`: Central Arabian Sea ($14.8^\circ	ext{N}-18.6^\circ	ext{N}, 59.4^\circ	ext{E}-65.0^\circ	ext{E}$, 2019–2022)
- `2902279`: Northern Bay of Bengal ($15.8^\circ	ext{N}-18.6^\circ	ext{N}, 89.2^\circ	ext{E}-93.3^\circ	ext{E}$, 2019–2021)
- `2902280`: Central Bay of Bengal ($13.9^\circ	ext{N}-16.8^\circ	ext{N}, 88.3^\circ	ext{E}-92.7^\circ	ext{E}$, 2019–2021)
- `2902283`: Southern Bay of Bengal ($12.9^\circ	ext{N}-16.6^\circ	ext{N}, 84.3^\circ	ext{E}-88.9^\circ	ext{E}$, 2019–2021)
- `1902670`: Equatorial North Indian Ocean / Sri Lanka ($3.9^\circ	ext{N}-12.6^\circ	ext{N}, 77.7^\circ	ext{E}-89.8^\circ	ext{E}$, 2023–2026)
- `1902672`: North-Central Arabian Sea ($15.5^\circ	ext{N}-19.0^\circ	ext{N}, 64.6^\circ	ext{E}-71.7^\circ	ext{E}$, 2023–2026)
- `1902673`: South-Central Arabian Sea ($10.7^\circ	ext{N}-14.5^\circ	ext{N}, 68.0^\circ	ext{E}-71.1^\circ	ext{E}$, 2023–2026)

In [3]:
GDAC_INCOIS_BASE = "https://data-argo.ifremer.fr/dac/incois"

NEW_ARGO_TARGETS = [
    {"wmo": "2902268", "basin": "Central Arabian Sea", "span": "2019-2022"},
    {"wmo": "2902279", "basin": "Northern Bay of Bengal", "span": "2019-2021"},
    {"wmo": "2902280", "basin": "Central Bay of Bengal", "span": "2019-2021"},
    {"wmo": "2902283", "basin": "Southern Bay of Bengal", "span": "2019-2021"},
    {"wmo": "1902670", "basin": "Equatorial NIO / Sri Lanka", "span": "2023-2026"},
    {"wmo": "1902672", "basin": "North-Central Arabian Sea", "span": "2023-2026"},
    {"wmo": "1902673", "basin": "South-Central Arabian Sea", "span": "2023-2026"},
]

print("=" * 80)
print("DOWNLOADING ADDITIONAL ARGO FLOATS FROM INCOIS GDAC")
print("=" * 80)

downloaded_argo_paths = []
for target in NEW_ARGO_TARGETS:
    wmo = target["wmo"]
    url = f"{GDAC_INCOIS_BASE}/{wmo}/{wmo}_prof.nc"
    dest = ARGO_DIR / f"{wmo}_prof.nc"
    desc = f"Argo Float {wmo} ({target['basin']}, {target['span']})"
    p = download_file_safely(url, dest, desc)
    if p:
        downloaded_argo_paths.append(p)

print(f"\nTotal Argo Files in Directory: {len(list(ARGO_DIR.glob('*_prof.nc')))}")

DOWNLOADING ADDITIONAL ARGO FLOATS FROM INCOIS GDAC
  [+] Downloading Argo Float 2902268 (Central Arabian Sea, 2019-2022)...
      URL: https://data-argo.ifremer.fr/dac/incois/2902268/2902268_prof.nc


      [SUCCESS] Saved: 2902268_prof.nc (2368.7 KB)
  [+] Downloading Argo Float 2902279 (Northern Bay of Bengal, 2019-2021)...
      URL: https://data-argo.ifremer.fr/dac/incois/2902279/2902279_prof.nc


      [SUCCESS] Saved: 2902279_prof.nc (1367.9 KB)
  [+] Downloading Argo Float 2902280 (Central Bay of Bengal, 2019-2021)...
      URL: https://data-argo.ifremer.fr/dac/incois/2902280/2902280_prof.nc


      [SUCCESS] Saved: 2902280_prof.nc (1801.1 KB)
  [+] Downloading Argo Float 2902283 (Southern Bay of Bengal, 2019-2021)...
      URL: https://data-argo.ifremer.fr/dac/incois/2902283/2902283_prof.nc


      [SUCCESS] Saved: 2902283_prof.nc (1848.1 KB)
  [+] Downloading Argo Float 1902670 (Equatorial NIO / Sri Lanka, 2023-2026)...
      URL: https://data-argo.ifremer.fr/dac/incois/1902670/1902670_prof.nc


      [SUCCESS] Saved: 1902670_prof.nc (827.1 KB)
  [+] Downloading Argo Float 1902672 (North-Central Arabian Sea, 2023-2026)...
      URL: https://data-argo.ifremer.fr/dac/incois/1902672/1902672_prof.nc


      [SUCCESS] Saved: 1902672_prof.nc (819.9 KB)
  [+] Downloading Argo Float 1902673 (South-Central Arabian Sea, 2023-2026)...
      URL: https://data-argo.ifremer.fr/dac/incois/1902673/1902673_prof.nc


      [SUCCESS] Saved: 1902673_prof.nc (800.9 KB)

Total Argo Files in Directory: 13


## 2. Additional Satellite Suite: Winter Monsoon (January 15–18, 2024)

We download synchronous satellite observations for a 3rd seasonal epoch: **Northeast Winter Monsoon (January 15–18, 2024)** matching the exact $0.25^\circ$ North Indian Ocean grid ($0.125^\circ	ext{N}-24.875^\circ	ext{N}, 50.125^\circ	ext{E}-99.875^\circ	ext{E}$):
- **SST**: NOAA Coral Reef Watch CoralTemp v3.1 Daily ($0.05^\circ$ sampled at $0.25^\circ$)
- **SLA**: NOAA CoastWatch Daily Sea Level Anomaly ($0.25^\circ$)
- **Currents**: NOAA CoastWatch Absolute Geostrophic Surface Velocity ($u, v$ in m/s)

In [4]:
JAN2024_TIME = "[(2024-01-15T00:00:00Z):1:(2024-01-18T00:00:00Z)]"
JAN2024_TIME_SST = "[(2024-01-15T12:00:00Z):1:(2024-01-18T12:00:00Z)]"
LAT_BOUNDS = "[(0.125):1:(24.875)]"
LON_BOUNDS = "[(50.125):1:(99.875)]"

UPWELL_BASE = "https://upwell.pfeg.noaa.gov/erddap/griddap"
OCEANWATCH_BASE = "https://oceanwatch.pifsc.noaa.gov/erddap/griddap"

print("=" * 80)
print("DOWNLOADING WINTER MONSOON SATELLITE SUITE (JANUARY 15-18, 2024)")
print("=" * 80)

# 1. SST
sst_jan24_url = f"{OCEANWATCH_BASE}/CRW_sst_v3_1.nc?analysed_sst{JAN2024_TIME_SST}[(0.125):5:(24.875)][(50.125):5:(99.875)]"
sst_jan24_path = SST_DIR / "sst_north_indian_ocean_20240115_20240118.nc"
download_file_safely(sst_jan24_url, sst_jan24_path, "NOAA CoralTemp SST (Jan 15-18, 2024)")

# 2. SLA
sla_jan24_url = f"{UPWELL_BASE}/nesdisSSH1day.nc?sla{JAN2024_TIME}{LAT_BOUNDS}{LON_BOUNDS}"
sla_jan24_path = SLA_DIR / "sla_north_indian_ocean_20240115_20240118.nc"
download_file_safely(sla_jan24_url, sla_jan24_path, "NOAA CoastWatch SLA (Jan 15-18, 2024)")

# 3. Currents
curr_jan24_url = f"{UPWELL_BASE}/nesdisSSH1day.nc?ugos{JAN2024_TIME}{LAT_BOUNDS}{LON_BOUNDS},vgos{JAN2024_TIME}{LAT_BOUNDS}{LON_BOUNDS}"
curr_jan24_path = CURRENTS_DIR / "currents_north_indian_ocean_20240115_20240118.nc"
download_file_safely(curr_jan24_url, curr_jan24_path, "NOAA CoastWatch Geostrophic Currents (Jan 15-18, 2024)")

DOWNLOADING WINTER MONSOON SATELLITE SUITE (JANUARY 15-18, 2024)
  [+] Downloading NOAA CoralTemp SST (Jan 15-18, 2024)...
      URL: https://oceanwatch.pifsc.noaa.gov/erddap/griddap/CRW_sst_v3_1.nc?analysed_sst[(2024-01-15T12:00:00Z):1:(2024-01-18T12:00:00Z)][(0.125):5:(24.875)][(50.125):5:(99.875)]


      [SUCCESS] Saved: sst_north_indian_ocean_20240115_20240118.nc (637.5 KB)
  [+] Downloading NOAA CoastWatch SLA (Jan 15-18, 2024)...
      URL: https://upwell.pfeg.noaa.gov/erddap/griddap/nesdisSSH1day.nc?sla[(2024-01-15T00:00:00Z):1:(2024-01-18T00:00:00Z)][(0.125):1:(24.875)][(50.125):1:(99.875)]


      [SUCCESS] Saved: sla_north_indian_ocean_20240115_20240118.nc (633.1 KB)
  [+] Downloading NOAA CoastWatch Geostrophic Currents (Jan 15-18, 2024)...
      URL: https://upwell.pfeg.noaa.gov/erddap/griddap/nesdisSSH1day.nc?ugos[(2024-01-15T00:00:00Z):1:(2024-01-18T00:00:00Z)][(0.125):1:(24.875)][(50.125):1:(99.875)],vgos[(2024-01-15T00:00:00Z):1:(2024-01-18T00:00:00Z)][(0.125):1:(24.875)][(50.125):1:(99.875)]


      [SUCCESS] Saved: currents_north_indian_ocean_20240115_20240118.nc (1258.7 KB)


WindowsPath('C:/Users/HP/OneDrive/Desktop(1)/Project395/backend/data/raw/currents/currents_north_indian_ocean_20240115_20240118.nc')

## 3. Comprehensive Quality Control Audit of the Entire Argo Library

We audit all 13 Argo float NetCDFs across the library, inspecting:
- Total profiles and operational date spans
- Geographic coverage bounds (Latitude & Longitude)
- QC flag distributions (`TEMP_QC`, `PRES_QC`, `PSAL_QC`)
- Sensor artifacts (negative pressures, impossible salinities, extreme temperatures)
- Valid in-situ measurement counts

In [5]:
all_argo_files = sorted(list(ARGO_DIR.glob("*_prof.nc")))
print(f"Auditing all {len(all_argo_files)} Argo float NetCDF datasets:\n")

argo_audit_results = []
total_library_profiles = 0
total_library_valid = 0

for af in all_argo_files:
    wmo = af.name.split("_")[0]
    with xr.open_dataset(af) as ds:
        juld = pd.to_datetime(ds['JULD'].values)
        valid_juld = juld[~pd.isna(juld)]
        lats = ds['LATITUDE'].values
        lons = ds['LONGITUDE'].values
        v_lats = lats[np.isfinite(lats) & (lats < 9000)]
        v_lons = lons[np.isfinite(lons) & (lons < 9000)]
        pres = ds['PRES'].values
        temp = ds['TEMP'].values
        psal = ds['PSAL'].values if 'PSAL' in ds else None
        temp_qc = ds['TEMP_QC'].values if 'TEMP_QC' in ds else None

        n_profiles = len(valid_juld)
        total_library_profiles += n_profiles

        # Count QC '1' or '2'
        if temp_qc is not None:
            if hasattr(temp_qc, 'decode'):
                t_qc_arr = np.array([x.decode('utf-8', errors='ignore') for x in temp_qc.flatten()])
            else:
                t_qc_arr = temp_qc.flatten().astype(str)
            good_qc_count = int(np.isin(t_qc_arr, ['1', '2']).sum())
            total_measurements = len(t_qc_arr)
            qc_pct = round((good_qc_count / total_measurements) * 100, 1) if total_measurements > 0 else 0.0
        else:
            qc_pct = 100.0

        # Physical validity count
        v_mask = (pres >= 0) & (pres < 2500) & (temp >= 1.0) & (temp <= 35.0) & np.isfinite(temp)
        valid_profiles_in_float = int((v_mask.sum(axis=1) >= 10).sum())
        total_library_valid += valid_profiles_in_float

        basin = "Arabian Sea" if float(v_lons.mean()) < 77.5 else "Bay of Bengal"

        argo_audit_results.append({
            "WMO": wmo,
            "Basin": basin,
            "Total Profiles": n_profiles,
            "Valid QC Profiles": valid_profiles_in_float,
            "Start Date": valid_juld.min().strftime("%Y-%m-%d"),
            "End Date": valid_juld.max().strftime("%Y-%m-%d"),
            "Lat Range": f"{v_lats.min():.2f}° to {v_lats.max():.2f}°",
            "Lon Range": f"{v_lons.min():.2f}° to {v_lons.max():.2f}°",
            "QC Good (%)": qc_pct,
            "Max Depth (dbar)": round(float(pres[pres < 9000].max()), 1) if (pres < 9000).any() else 0.0
        })

df_argo_audit = pd.DataFrame(argo_audit_results)
print(df_argo_audit.to_string(index=False))
print(f"\nTotal In-Situ Profiles Across Library: {total_library_profiles}")
print(f"Total Profiles Passing Strict QC:     {total_library_valid} ({round(total_library_valid/total_library_profiles*100, 1)}%)")

Auditing all 13 Argo float NetCDF datasets:



    WMO         Basin  Total Profiles  Valid QC Profiles Start Date   End Date        Lat Range        Lon Range  QC Good (%)  Max Depth (dbar)
1902669 Bay of Bengal             110                108 2023-09-19 2026-10-23 13.05° to 17.02° 85.61° to 91.26°         88.9            2032.5
1902670 Bay of Bengal             112                111 2023-09-26 2026-10-30  3.88° to 12.60° 77.65° to 89.75°         84.4            2060.6
1902671   Arabian Sea             109                108 2023-10-09 2026-10-13 16.15° to 19.08° 67.78° to 70.95°         87.0            2031.8
1902672   Arabian Sea             111                111 2023-10-10 2026-10-24 15.52° to 18.95° 64.55° to 71.67°         86.9            2058.0
1902673   Arabian Sea             109                109 2023-10-26 2026-10-30 10.68° to 14.47° 67.97° to 71.08°         92.3            2036.2
2902264 Bay of Bengal             268                268 2018-04-05 2023-04-19  9.32° to 16.73° 82.20° to 90.66°         99.2           

## 4. Multi-Epoch Collocation Matching

We evaluate satellite–Argo collocation matching across all **13 Argo floats** and **3 distinct seasonal observation windows**:
- **Epoch 1: Late October 2019** (Post-Monsoon Transition)
- **Epoch 2: Mid January 2024** (Northeast Winter Monsoon)
- **Epoch 3: Early May 2024** (Pre-Monsoon Tropical Warming)

### Collocation Tolerance Tiers:
- **`Tier 0: Exact Window Match`**: $\Delta t = 0\text{ days}$ (Surfaced on satellite observation day)
- **`Tier 1: Synoptic Match`**: $|\Delta t| \le 1.0\text{ day}$ ($24\text{ hrs}$)
- **`Tier 2: Standard Match`**: $1.0 < |\Delta t| \le 3.0\text{ days}$ ($72\text{ hrs}$)
- **`Tier 3: Mesoscale Match`**: $3.0 < |\Delta t| \le 5.0\text{ days}$ ($120\text{ hrs}$)

In [6]:
OBSERVATION_EPOCHS = [
    {
        "epoch_id": "Epoch_1_Oct2019",
        "name": "Late October 2019 (Post-Monsoon)",
        "start": pd.to_datetime("2019-10-28 00:00:00"),
        "end": pd.to_datetime("2019-10-31 23:59:59"),
        "sst_file": SST_DIR / "sst_north_indian_ocean_20191028_20191031.nc",
        "sla_file": SLA_DIR / "sla_north_indian_ocean.nc",
        "curr_file": CURRENTS_DIR / "currents_north_indian_ocean.nc",
    },
    {
        "epoch_id": "Epoch_2_Jan2024",
        "name": "Mid January 2024 (Winter Monsoon)",
        "start": pd.to_datetime("2024-01-15 00:00:00"),
        "end": pd.to_datetime("2024-01-18 23:59:59"),
        "sst_file": SST_DIR / "sst_north_indian_ocean_20240115_20240118.nc",
        "sla_file": SLA_DIR / "sla_north_indian_ocean_20240115_20240118.nc",
        "curr_file": CURRENTS_DIR / "currents_north_indian_ocean_20240115_20240118.nc",
    },
    {
        "epoch_id": "Epoch_3_May2024",
        "name": "Early May 2024 (Pre-Monsoon)",
        "start": pd.to_datetime("2024-05-01 00:00:00"),
        "end": pd.to_datetime("2024-05-03 23:59:59"),
        "sst_file": SST_DIR / "sst_north_indian_ocean.nc",
        "sla_file": SLA_DIR / "sla_north_indian_ocean_20240501_20240503.nc",
        "curr_file": CURRENTS_DIR / "currents_north_indian_ocean_20240501_20240503.nc",
    },
]

collocation_matches = []

for af in all_argo_files:
    wmo = af.name.split("_")[0]
    with xr.open_dataset(af) as ds:
        juld = pd.to_datetime(ds['JULD'].values)
        lats = ds['LATITUDE'].values
        lons = ds['LONGITUDE'].values
        cycles = ds['CYCLE_NUMBER'].values
        pres = ds['PRES'].values
        temp = ds['TEMP'].values

        for epoch in OBSERVATION_EPOCHS:
            e_name = epoch["name"]
            e_id = epoch["epoch_id"]
            e_start = epoch["start"]
            e_end = epoch["end"]

            for i in range(len(juld)):
                t = juld[i]
                if pd.isna(t):
                    continue

                lat = float(lats[i])
                lon = float(lons[i])

                # Check North Indian Ocean bounds
                if not (0.0 <= lat <= 25.0 and 50.0 <= lon <= 100.0):
                    continue

                # Check profile physical validity
                p_p = pres[i]
                p_t = temp[i]
                v_pts = (p_p >= 0) & (p_p < 2500) & (p_t >= 1.0) & (p_t <= 35.0) & np.isfinite(p_t)
                if v_pts.sum() < 10:
                    continue

                # Calculate offset in days
                if t < e_start:
                    d_days = -(e_start - t).total_seconds() / 86400.0
                elif t > e_end:
                    d_days = (t - e_end).total_seconds() / 86400.0
                else:
                    d_days = 0.0

                abs_days = abs(d_days)

                if abs_days <= 5.0:
                    if abs_days == 0.0:
                        tier = "Tier 0: Exact Window Match"
                    elif abs_days <= 1.0:
                        tier = "Tier 1: Synoptic (<= 1.0 d)"
                    elif abs_days <= 3.0:
                        tier = "Tier 2: Standard (<= 3.0 d)"
                    else:
                        tier = "Tier 3: Mesoscale (<= 5.0 d)"

                    basin = "Arabian Sea" if lon < 77.5 else "Bay of Bengal"

                    collocation_matches.append({
                        "Epoch": e_id,
                        "Season / Period": e_name,
                        "Float WMO": wmo,
                        "Cycle": int(cycles[i]),
                        "Basin": basin,
                        "Timestamp": t.strftime("%Y-%m-%d %H:%M UTC"),
                        "Latitude": round(lat, 3),
                        "Longitude": round(lon, 3),
                        "Offset (hrs)": round(abs_days * 24.0, 1),
                        "Tier": tier,
                        "Depth Levels": int(v_pts.sum()),
                        "Surface Temp (°C)": round(float(p_t[v_pts][0]), 2),
                        "Deep Temp (°C)": round(float(p_t[v_pts][-1]), 2),
                    })

df_matches = pd.DataFrame(collocation_matches)
print(f"TOTAL VALID SATELLITE–ARGO MATCHED PROFILES: {len(df_matches)}\n")
print("Matches by Observation Epoch:")
print(df_matches['Season / Period'].value_counts())
print("\nMatches by Collocation Tier:")
print(df_matches['Tier'].value_counts())

TOTAL VALID SATELLITE–ARGO MATCHED PROFILES: 29

Matches by Observation Epoch:
Season / Period
Late October 2019 (Post-Monsoon)     16
Early May 2024 (Pre-Monsoon)          8
Mid January 2024 (Winter Monsoon)     5
Name: count, dtype: int64

Matches by Collocation Tier:
Tier
Tier 0: Exact Window Match      9
Tier 1: Synoptic (<= 1.0 d)     8
Tier 3: Mesoscale (<= 5.0 d)    7
Tier 2: Standard (<= 3.0 d)     5
Name: count, dtype: int64


In [7]:
# Display formatted table of all matched profiles
cols_show = ["Epoch", "Float WMO", "Cycle", "Basin", "Timestamp", "Offset (hrs)", "Tier", "Latitude", "Longitude", "Surface Temp (°C)", "Deep Temp (°C)"]
df_matches_sorted = df_matches.sort_values(by=["Epoch", "Offset (hrs)"])
print(df_matches_sorted[cols_show].to_string(index=False))

          Epoch Float WMO  Cycle         Basin            Timestamp  Offset (hrs)                         Tier  Latitude  Longitude  Surface Temp (°C)  Deep Temp (°C)
Epoch_1_Oct2019   2902267     28   Arabian Sea 2019-10-28 14:18 UTC           0.0   Tier 0: Exact Window Match    18.644     58.712              28.49            2.87
Epoch_1_Oct2019   2902268     28   Arabian Sea 2019-10-29 14:04 UTC           0.0   Tier 0: Exact Window Match    17.515     61.225              28.12            3.19
Epoch_1_Oct2019   2902280     31 Bay of Bengal 2019-10-28 14:08 UTC           0.0   Tier 0: Exact Window Match    16.467     92.342              30.55            2.71
Epoch_1_Oct2019   2902283     31 Bay of Bengal 2019-10-29 14:04 UTC           0.0   Tier 0: Exact Window Match    13.497     86.513              29.76            2.80
Epoch_1_Oct2019   2902264    115 Bay of Bengal 2019-10-27 17:24 UTC           6.6  Tier 1: Synoptic (<= 1.0 d)    13.943     87.570              30.01            2.9

## 5. Inspection Report Registry Update

We update `dataset_inspection_report.json` with the complete inventory of all raw NetCDF files across the library.

In [8]:
# Run inspection scan on all .nc files in data/raw
raw_nc_files = sorted(list(DATA_RAW.glob("**/*.nc")))
print(f"Total Raw NetCDF Files in Library: {len(raw_nc_files)}")

inspection_records = []
for f in raw_nc_files:
    rel = f.relative_to(DATA_RAW)
    sz_kb = round(f.stat().st_size / 1024, 2)
    with netCDF4.Dataset(f, "r") as nc:
        fmt = nc.data_model
        title = getattr(nc, "title", "N/A")
    with xr.open_dataset(f) as ds:
        dims = {dim: int(size) for dim, size in ds.sizes.items()}
        vars_list = list(ds.data_vars.keys())
    inspection_records.append({
        "filename": f.name,
        "relative_path": str(rel),
        "size_kb": sz_kb,
        "format": fmt,
        "title": title,
        "dimensions": dims,
        "variables": vars_list,
    })

rep_path = DATA_RAW / "dataset_inspection_report.json"
with open(rep_path, "w", encoding="utf-8") as f:
    json.dump(inspection_records, f, indent=2)

print(f"[SUCCESS] Updated {rep_path.name} with {len(inspection_records)} dataset records.")

Total Raw NetCDF Files in Library: 22


[SUCCESS] Updated dataset_inspection_report.json with 22 dataset records.
